<a href="https://colab.research.google.com/github/abrandonwang/text-to-sound/blob/main/latent_diffusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/data
!cp /content/drive/MyDrive/text-to-sound/data/* /content/data/
!ls -lh /content/data

latents = "/content/drive/MyDrive/text-to-sound/data/latents.npy"
clap_audio = "/content/drive/MyDrive/text-to-sound/data/clap_audio.npy"
CKPT = "/content/drive/MyDrive/text-to-sound/checkpoints"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 1.4G
-rw------- 1 root root 1.3M Nov  7 22:20 captions_dev.csv
-rw------- 1 root root  25M Nov  7 22:20 clap_audio.npy
-rw------- 1 root root 125M Nov  7 22:20 clap_text.npy
-rw------- 1 root root  132 Nov  7 22:20 latent_scale.npy
-rw------- 1 root root 249M Nov  7 22:20 latents.npy
-rw------- 1 root root 995M Nov  7 22:21 mels.npy
-rw------- 1 root root  136 Nov  7 22:21 mel_stats.npy
-rw------- 1 root root 952K Nov  7 22:21 meta.csv


In [3]:
import torch
import glob
import pandas as pd
import os
from IPython.display import Audio, display
import librosa, librosa.display
import matplotlib.pyplot as plt
import numpy as np
import random
!test -d BigVGAN || git clone -q https://github.com/NVIDIA/BigVGAN
!sed -i 's/proxies: Optional\[Dict\],/proxies: Optional[Dict] = None,/; s/resume_download: bool,/resume_download: bool = False,/' BigVGAN/bigvgan.py
!grep -n "proxies\|resume_download" BigVGAN/bigvgan.py
import sys; sys.path.insert(0, "BigVGAN")
import torch, bigvgan
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader, Subset
import torch.nn.functional as F
import math
import torch.nn as nn
from transformers import ClapModel, ClapProcessor
%matplotlib inline
device = "cuda"
vocoder = bigvgan.BigVGAN.from_pretrained("nvidia/bigvgan_v2_22khz_80band_256x", use_cuda_kernel = False)
vocoder.remove_weight_norm()
vocoder = vocoder.eval().to(device)
print(vocoder.h.sampling_rate)

DATA = "/content/data/"
mels = np.load(DATA + "mels.npy", mmap_mode = "r")
stats = np.load(DATA + "mel_stats.npy")
MIN, MAX = stats[0], stats[1]
meta = pd.read_csv(DATA + "meta.csv")

422:        proxies: Optional[Dict] = None,
423:        resume_download: bool = False,
444:                proxies=proxies,
445:                resume_download=resume_download,
476:                proxies=proxies,
477:                resume_download=resume_download,


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `hf_hub_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading weights from nvidia/bigvgan_v2_22khz_80band_256x
Removing weight norm...
22050


In [7]:
class Autoencoder(nn.Module):
  def __init__(self):
    super().__init__()
    self.encoder = nn.Sequential(
      nn.Conv2d(1, 64, 3, 1, 1),
      nn.SiLU(),
      nn.Conv2d(64, 64, 3, 1, 1),
      nn.SiLU(),
      nn.Conv2d(64, 128, 3, 2, 1),
      nn.SiLU(),
      nn.Conv2d(128, 128, 3, 1, 1),
      nn.SiLU(),
      nn.Conv2d(128, 256, 3, 2, 1),
      nn.SiLU(),
      nn.Conv2d(256, 256, 3, 1, 1),
      nn.SiLU(),
      nn.Conv2d(256, 8, 3, 1, 1),
    )
    self.decoder = nn.Sequential(
        nn.Conv2d(4, 256, 3, 1, 1),
        nn.SiLU(),
        nn.Conv2d(256, 256, 3, 1, 1),
        nn.SiLU(),
        nn.Upsample(scale_factor = 2),
        nn.Conv2d(256, 128, 3, 1, 1),
        nn.SiLU(),
        nn.Conv2d(128, 128, 3, 1, 1),
        nn.SiLU(),
        nn.Upsample(scale_factor = 2),
        nn.Conv2d(128, 64, 3, 1, 1),
        nn.SiLU(),
        nn.Conv2d(64, 64, 3, 1, 1),
        nn.SiLU(),
        nn.Conv2d(64, 1, 3, 1, 1),
    )
  def forward(self, x):
    mu, logvar = self.encoder(x).chunk(2, dim = 1)
    z = mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
    return self.decoder(z), mu, logvar

model = Autoencoder().to(device)
model.load_state_dict(torch.load(CKPT + "/vae_gan2.pt"))

<All keys matched successfully>

In [13]:
class LatentDataset(Dataset):
  def __init__(self, split):
    self.rows = meta.index[meta["split"] == split].to_numpy()
    self.latents = np.load(latents)
    self.clap_audio = np.load(clap_audio)
    self.scale = 0.6863

  def __len__(self):
    return len(self.rows)

  def __getitem__(self, i):
    row = self.rows[i]
    latent = torch.tensor(self.latents[row], dtype = torch.float32) * self.scale
    clap = torch.tensor(self.clap_audio[row], dtype = torch.float32) * self.scale
    clap = F.normalize(clap, dim = 0)
    return latent, clap

In [14]:
train_data = LatentDataset("train")
loader = DataLoader(train_data, batch_size = 32, shuffle = True, num_workers = 2)
lat, clap = next(iter(loader))
print(len(train_data), lat.shape, clap.shape, lat.std().item())

22848 torch.Size([32, 4, 20, 64]) torch.Size([32, 512]) 0.987897515296936
